In [ ]:
import os
import numpy as np
import re  # 新增正则模块

# 新增数字提取函数
def extract_number(filename):
    """从文件名中提取末尾的数字（支持任意前缀）"""
    match = re.search(r'_(\d+)\.txt$', filename)
    return int(match.group(1)) if match else -1

data_dict = {
    "c": [],        # c_*.txt
    "p": [],        # p_*.txt
    "v": [],        # v_*.txt
    "bdry_points": [],  # bdry_points_*.txt
    "ctrl_points": [],  # ctrl_points_*.txt
    "ctrl_nxny": []
}

base_dirs = ["functions", "points"]

for base_dir in base_dirs:
    if not os.path.exists(base_dir):
        continue
    
    # 修改排序方式：按数字大小排序
    for filename in sorted(os.listdir(base_dir), key=extract_number):  # 关键修改
        filepath = os.path.join(base_dir, filename)
        
        if filename.startswith("c_"):
            data_dict["c"].append(np.loadtxt(filepath))
        elif filename.startswith("p_"):
            data_dict["p"].append(np.loadtxt(filepath))
        elif filename.startswith("v_"):
            data_dict["v"].append(np.loadtxt(filepath))
        elif filename.startswith("bdry_points_"):
            data_dict["bdry_points"].append(np.loadtxt(filepath))
        elif filename.startswith("ctrl_points_"):
            data_dict["ctrl_points"].append(np.loadtxt(filepath))
        elif filename.startswith("ctrl_nxny_"):
            data_dict["ctrl_nxny"].append(np.loadtxt(filepath))

c_list = [np.array(arr) for arr in data_dict["c"]]
p_list = [np.array(arr) for arr in data_dict["p"]]
v_list = [np.array(arr) for arr in data_dict["v"]]
bdry_points_list = [np.array(arr) for arr in data_dict["bdry_points"]]
ctrl_points_list = [np.array(arr) for arr in data_dict["ctrl_points"]]
ctrl_nxny_list = [np.array(arr) for arr in data_dict["ctrl_nxny"]]

print(f"c_list: {len(c_list)}, p_list: {len(p_list)}, v_list: {len(v_list)}, bdry_points_list: {len(bdry_points_list)}, ctrl_points_list: {len(ctrl_points_list)}, ctrl_nxny_list: {len(ctrl_nxny_list)}")

In [ ]:
steps = min(len(c_list), len(p_list), len(v_list), len(bdry_points_list), len(ctrl_points_list), len(ctrl_nxny_list))

In [ ]:
x_min = -10
y_min = -10
x_max = 10
y_max = 10

In [ ]:
import matplotlib.pyplot as plt

# for t in range(steps):
#     fig, axes = plt.subplots(1, 3, figsize=(18, 6), dpi = 600)  

#     ax0 = axes[0]
#     bdry_points = bdry_points_list[t]  
#     ctrl_points = ctrl_points_list[t] 

#     # velocity = np.zeros((len(ctrl_points), 2))
#     # for i in range(len(ctrl_points)):
#     #     velocity[i, 0] = v_list[t][i] * ctrl_nxny_list[t][i, 0]
#     #     velocity[i, 1] = v_list[t][i] * ctrl_nxny_list[t][i, 1]

#     # ax0.scatter(bdry_points[:, 0], bdry_points[:, 1], color='black', label="Boundary Points", s=1)
#     ax0.scatter(ctrl_points[:, 0], ctrl_points[:, 1], color='b', label="Control Points", s=22)
#     ax0.set_title(f"Control Points at Time t = {t * 0.01}")
#     ax0.set_xlim([x_min, x_max])
#     ax0.set_ylim([y_min, y_max])
#     # ax0.quiver(ctrl_points[:, 0], ctrl_points[:, 1], velocity[:, 0], velocity[:, 1], angles="xy", scale_units="xy", scale=1, width = 0.004, color='red', label = 'Current Velocity')
#     ax0.legend()
#     ax0.set_aspect("equal") 
#     ax0.grid(True)
#     # ax0.axis("equal")

#     im1 = axes[1].imshow(c_list[t].T, extent=[x_min, x_max, y_min, y_max], origin='lower', cmap='viridis')
#     axes[1].scatter(bdry_points[:, 0], bdry_points[:, 1], s=1, c='w', label='Boundary Points')  # Add bdry_points
#     axes[1].set_title(f'Nutrient Concentration at Time t = {t * 0.01}')
#     # axes[1].legend()
#     axes[1].set_aspect('equal')
#     fig.colorbar(im1, ax=axes[1], shrink=0.8)

#     im2 = axes[2].imshow(p_list[t].T, extent=[x_min, x_max, y_min, y_max], origin='lower', cmap='viridis')
#     axes[2].scatter(bdry_points[:, 0], bdry_points[:, 1], s=1, c='w', label='Boundary Points')  # Add bdry_points
#     axes[2].set_title(f'Limit Pressure at Time t = {t * 0.01}')
#     # axes[2].legend()
#     axes[2].set_aspect('equal')
#     fig.colorbar(im2, ax=axes[2], shrink=0.8)

#     plt.tight_layout()
#     plt.show()

In [ ]:
from scipy.interpolate import splprep, splev
import matplotlib.pyplot as plt

def plot_control_points_steps(ctrl_points_list, steps_to_plot):
    """
    Plot control points at selected time steps on the same figure.
    
    Parameters:
        ctrl_points_list : list of np.array
            List containing control points for each time step, shape (N, 2)
        steps_to_plot : list of int
            List of time step indices to plot
    """
    plt.figure(figsize=(8, 8), dpi=600)
    
    colors = plt.cm.viridis(np.linspace(0, 1, len(steps_to_plot)))  
    # colors = plt.cm.plasma(np.linspace(0, 1, len(steps_to_plot)))
    
    for idx, t in enumerate(steps_to_plot):
        ctrl_points = ctrl_points_list[t]
        
        closed_points = np.vstack([ctrl_points, ctrl_points[0]])
        
        tck, _ = splprep([closed_points[:, 0], closed_points[:, 1]], s=0, per=True)
        smooth = np.linspace(0, 1, 400)
        x_smooth, y_smooth = splev(smooth, tck)
        
        plt.plot(x_smooth, y_smooth, color=colors[idx], label=f"t = {round(t * 0.01, 2)}")
        plt.scatter(ctrl_points[:, 0], ctrl_points[:, 1], color=colors[idx], s=10)
    
    plt.xlim([x_min, x_max])
    plt.ylim([y_min, y_max])
    plt.gca().set_aspect("equal")
    plt.grid(True)
    plt.legend()
    plt.title("Control Points and Tumor Boundaries at Selected Time Steps")
    # plt.title("Tumor Boundaries at Selected Time Steps")
    plt.show()

In [ ]:
plot_control_points_steps(ctrl_points_list, [0, 10, 20, 30, 40, 50, 60, 70, 80, 90])